# Subtype Characterization — What Makes Each Subtype Distinct?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/08_characterization.ipynb)

**What this does:** After discovering subtypes, characterizes them by identifying enriched pathways, top contributing genes, effect sizes, and generates publication-ready heatmaps.

**Analyses included:**
- Pathway enrichment (Kruskal-Wallis + FDR correction)
- Gene-level contribution scores (Cohen's d effect sizes)
- Subtype profile summaries
- Pathway and gene heatmaps
- Export to CSV/Excel

**Prerequisites:** [00_quick_demo.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/00_quick_demo.ipynb)

In [ ]:
# Install pathway-subtyping
!pip install -q pathway-subtyping==0.3.0

import pathway_subtyping
print(f"pathway-subtyping v{pathway_subtyping.__version__}")

## 1. Generate and Cluster Data

In [ ]:
from pathway_subtyping import (
    SimulationConfig,
    generate_synthetic_data,
    run_clustering,
    ClusteringAlgorithm,
)
import numpy as np

sim = generate_synthetic_data(SimulationConfig(
    n_samples=200,
    n_pathways=10,
    n_genes_per_pathway=20,
    n_subtypes=3,
    effect_size=1.5,
    noise_level=1.0,
    seed=42,
))

clustering = run_clustering(
    sim.pathway_scores.values,
    n_clusters=3,
    algorithm=ClusteringAlgorithm.GMM,
    seed=42,
)

print(f"Data: {sim.pathway_scores.shape[0]} samples \u00d7 {sim.pathway_scores.shape[1]} pathways")
print(f"Clusters: {len(set(clustering.labels))} subtypes")
print(f"Cluster sizes: {dict(zip(*np.unique(clustering.labels, return_counts=True)))}")

## 2. Full Characterization

`characterize_subtypes()` runs all analyses at once: pathway enrichment, gene contributions, and subtype profiles.

In [ ]:
from pathway_subtyping import characterize_subtypes

result = characterize_subtypes(
    pathway_scores=sim.pathway_scores,
    cluster_labels=clustering.labels,
    gene_burdens=sim.gene_burdens,
    pathways=sim.pathways,
    fdr_threshold=0.05,
)

print(f"Subtypes characterized: {result.n_subtypes}")
print(f"Significant pathways:   {result.n_significant_pathways}")
print(f"\nSubtype profiles:")
for profile in result.subtype_profiles:
    top = profile.top_pathways[:3] if profile.top_pathways else []
    print(f"  Subtype {profile.subtype_id}: {profile.n_samples} samples, "
          f"top pathways: {top}")

## 3. Pathway Enrichment Analysis

Which pathways are significantly different across subtypes? Uses Kruskal-Wallis test with Benjamini-Hochberg FDR correction.

In [ ]:
from pathway_subtyping import pathway_enrichment_analysis

enrichments = pathway_enrichment_analysis(
    sim.pathway_scores,
    clustering.labels,
    fdr_threshold=0.05,
)

print(f"{'Pathway':<20} {'H-stat':>8} {'FDR':>10} {'Significant':>12}")
print("-" * 52)
for e in sorted(enrichments, key=lambda x: x.fdr_pvalue):
    sig = "***" if e.is_significant else ""
    print(f"{e.pathway:<20} {e.h_statistic:>8.2f} {e.fdr_pvalue:>10.4f} {sig:>12}")

## 4. Gene-Level Contributions

Which genes contribute most to each subtype's distinction? Uses Cohen's d effect size (subtype vs rest).

In [ ]:
from pathway_subtyping import gene_contribution_scores

contributions = gene_contribution_scores(
    sim.gene_burdens,
    clustering.labels,
    top_n=10,
)

# Show top genes for each subtype
for subtype_id, genes in contributions.items():
    print(f"\nSubtype {subtype_id} \u2014 Top 10 Contributing Genes:")
    print(f"  {'Gene':<15} {'Cohen d':>10} {'Mean (subtype)':>15} {'Mean (rest)':>12}")
    print(f"  {'-'*55}")
    for g in genes[:10]:
        print(f"  {g.gene:<15} {g.effect_size:>10.3f} {g.mean_in_subtype:>15.3f} {g.mean_in_rest:>12.3f}")

## 5. Pathway Heatmap

Visualize mean pathway scores per subtype. Red = elevated, blue = depleted.

In [ ]:
from pathway_subtyping import generate_subtype_heatmap

fig = generate_subtype_heatmap(
    sim.pathway_scores,
    clustering.labels,
    title="Pathway Profiles by Subtype",
)

## 6. Gene Heatmap

Visualize the top contributing genes per subtype.

In [ ]:
from pathway_subtyping import generate_gene_heatmap

fig = generate_gene_heatmap(
    sim.gene_burdens,
    clustering.labels,
    top_n_genes=15,
    title="Top Contributing Genes by Subtype",
)

## 7. Export Results

Save characterization results to CSV or Excel for further analysis or publication.

In [ ]:
from pathway_subtyping import export_characterization

# Export to CSV files
export_characterization(result, output_dir="characterization_output", format="csv")
print("Exported CSV files to characterization_output/")

# Also available: result.to_dict() for programmatic access
summary = result.to_dict()
print(f"\nResult keys: {list(summary.keys())}")
print(f"Subtype profiles: {len(summary['subtype_profiles'])}")
print(f"Enrichment results: {len(summary['enrichment_results'])}")

## 8. Combining with Visualization

Use the visualization module for interactive characterization views.

In [ ]:
# Interactive heatmap (requires pip install pathway-subtyping[viz])
try:
    from pathway_subtyping import plot_interactive_heatmap, plot_subtype_trajectories

    fig = plot_interactive_heatmap(sim.pathway_scores, clustering.labels, title="Interactive Pathway Profiles")
    fig.show()

    fig = plot_subtype_trajectories(sim.pathway_scores, clustering.labels, top_n=8)
    fig.show()
except ImportError:
    print("Install pathway-subtyping[viz] for interactive charts")
    print("Static heatmaps (above) work with the base install")

## Summary

| Analysis | Function | Output |
|----------|----------|--------|
| Full characterization | `characterize_subtypes()` | `CharacterizationResult` with all below |
| Pathway enrichment | `pathway_enrichment_analysis()` | List of `PathwayEnrichment` per pathway |
| Gene contributions | `gene_contribution_scores()` | Dict of `GeneContribution` per subtype |
| Pathway heatmap | `generate_subtype_heatmap()` | matplotlib figure |
| Gene heatmap | `generate_gene_heatmap()` | matplotlib figure |
| Export | `export_characterization()` | CSV/Excel files |

## Next Steps

- **Benchmark methods:** Use `run_benchmark_comparison()` to compare with NMF, PCA+K-means
- **Sensitivity testing:** [07_sensitivity_analysis.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/07_sensitivity_analysis.ipynb)
- **API reference:** [Characterization API](https://github.com/topmist-admin/pathway-subtyping-framework/blob/main/docs/api/characterization.md)

---
*Built with [pathway-subtyping](https://pypi.org/project/pathway-subtyping/). Disease-agnostic. Open source.*